In [1]:
%pip install ContrailOnlineCAClient

  Using cached ContrailOnlineCAClient-0.5.1-py3-none-any.whl.metadata (2.7 kB)
Using cached ContrailOnlineCAClient-0.5.1-py3-none-any.whl (197 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
# encoding: utf-8
"""
remote_nc_reader.py (Jupyter Version)
===================

Python script for downloading a file from the CEDA archive, adapted for JupyterLab.

Usage:
```
download_nc_file("<url>")
```

Example:
```
URL = "http://dap.ceda.ac.uk/thredds/dodsC/badc/ukcp18/data/marine-sim/skew-trend/rcp85/skewSurgeTrend/latest/skewSurgeTrend_marine-sim_rcp85_trend_2007-2099.nc"
download_nc_file(URL)
```
"""

'\nremote_nc_reader.py (Jupyter Version)\n===================\n\nPython script for downloading a file from the CEDA archive, adapted for JupyterLab.\n\nUsage:\n```\ndownload_nc_file("<url>")\n```\n\nExample:\n```\nURL = "http://dap.ceda.ac.uk/thredds/dodsC/badc/ukcp18/data/marine-sim/skew-trend/rcp85/skewSurgeTrend/latest/skewSurgeTrend_marine-sim_rcp85_trend_2007-2099.nc"\ndownload_nc_file(URL)\n```\n'

In [3]:
# Import standard libraries
import os
import datetime
import requests

In [4]:
# Import third-party libraries
from cryptography import x509
from cryptography.hazmat.backends import default_backend
from contrail.security.onlineca.client import OnlineCaClient

In [5]:
CERTS_DIR = os.path.expanduser('~/.certs')
if not os.path.isdir(CERTS_DIR):
    os.makedirs(CERTS_DIR)

TRUSTROOTS_DIR = os.path.join(CERTS_DIR, 'ca-trustroots')
CREDENTIALS_FILE_PATH = os.path.join(CERTS_DIR, 'credentials.pem')

TRUSTROOTS_SERVICE = 'https://slcs.ceda.ac.uk/onlineca/trustroots/'
CERT_SERVICE = 'https://slcs.ceda.ac.uk/onlineca/certificate/'

In [6]:
def cert_is_valid(cert_file, min_lifetime=0):
    """
    Returns boolean - True if the certificate is in date.
    Optional argument min_lifetime is the number of seconds
    which must remain.

    :param cert_file: certificate file path.
    :param min_lifetime: minimum lifetime (seconds)
    :return: boolean
    """
    try:
        with open(cert_file, 'rb') as f:
            crt_data = f.read()
    except IOError:
        return False

    try:
        cert = x509.load_pem_x509_certificate(crt_data, default_backend())
    except ValueError:
        return False

    now = datetime.datetime.now()
    return (cert.not_valid_before <= now
            and cert.not_valid_after > now + datetime.timedelta(0, min_lifetime))

In [7]:
def setup_credentials():
    """
    Download and create required credentials files.

    Return True if credentials were set up.
    Return False if credentials were already set up.
    """
    if cert_is_valid(CREDENTIALS_FILE_PATH):
        print('[INFO] Security credentials already set up.')
        return False

    username = 'tnobrega'
    password = 'x9kp2HU1pX'

    if not username or not password:
        raise ValueError("CEDA_USERNAME and CEDA_PASSWORD environment variables are required")

    onlineca_client = OnlineCaClient()
    onlineca_client.ca_cert_dir = TRUSTROOTS_DIR

    trustroots = onlineca_client.get_trustroots(
        TRUSTROOTS_SERVICE,
        bootstrap=True,
        write_to_ca_cert_dir=True)

    key_pair, certs = onlineca_client.get_certificate(
        username,
        password,
        CERT_SERVICE,
        pem_out_filepath=CREDENTIALS_FILE_PATH)

    print('[INFO] Security credentials set up.')
    return True

In [8]:
def download_nc_file(file_url):
    """
    Download NetCDF file from the given URL.

    :param file_url: URL to the NetCDF file.
    """
    try:
        setup_credentials()
    except ValueError as e:
        print(e)
        return

    response = requests.get(file_url, cert=(CREDENTIALS_FILE_PATH), verify=False)
    filename = file_url.rsplit('/', 1)[-1]
    with open(filename, 'wb') as file_object:
        file_object.write(response.content)
    print(f"[INFO] File downloaded: {filename}")

In [9]:
setup_credentials()

/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'slcs.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


[INFO] Security credentials set up.


True

In [10]:
download_nc_file('https://dap.ceda.ac.uk/badc/faam/data/2012/b747-oct-01/core_processed/')

[INFO] Security credentials already set up.


/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dap.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: ''

In [11]:
url ='https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core-cloud-phy_faam_20120914_v500_r0_b731.nc'
download_nc_file(url)

[INFO] Security credentials already set up.


/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dap.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'auth.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'auth.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/srv/conda/envs/notebook/l

[INFO] File downloaded: core-cloud-phy_faam_20120914_v500_r0_b731.nc
